# Análise de Vendas Liah — tendências, padrões de compra e granularidades (bigframes)
Análise descritiva de hoje para trás, só com **pedidos válidos** (sem cancelados, PIX não pago, aguardando pagamento e testes).

**O que mudou:** os dados vêm direto do BigQuery via **BigQuery DataFrames (`bigframes`)** — não é mais preciso exportar CSV.
A regra de limpeza está num lugar só (`liah_dados.py`), igual à usada no notebook de previsão. Nova seção 9 com os
cortes por **canal, origem, região/UF, grupo, subgrupo e marca** (de `gold_core.dim_liah_produtos`) e o ruído de cada nível.

**Organização da pasta:** na raiz ficam só os notebooks. `modulos/` (liah_dados.py, previsao_modelos.py), `dados/` (histórico e cache das consultas), `saida/` (resultados), `dashboard/` (painel), `docs/` (guia e SQL), `teste/` (dados fictícios).

> Análise interna sujeita a validação pela área responsável; não é número oficial da Tecsa.

In [ ]:
# Instala SÓ o que estiver faltando. Não atualiza bibliotecas que o Colab já traz (numpy, pandas, pyarrow, scikit-learn...):
# atualizar biblioteca já carregada pode derrubar o kernel ("kernel restarted") sem mensagem de erro.
import importlib.util, subprocess, sys
faltam = [p for m, p in {'bigframes': 'bigframes', 'holidays': 'holidays'}.items() if importlib.util.find_spec(m) is None]
if faltam:
    print('Instalando:', faltam)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltam], check=True)
else:
    print('Todas as bibliotecas já estão instaladas.')

In [ ]:
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.ticker as mt, holidays

plt.rcParams.update({'figure.figsize': (13, 4.5), 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
AZUL, LARANJA, VERDE, CINZA = '#2a78d6', '#eb6834', '#1baf7a', '#a9b0a8'
CORES_ANO = {2024: VERDE, 2025: LARANJA, 2026: AZUL}
MESES = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']
DIAS = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
reais = mt.FuncFormatter(lambda v, _: f'R$ {v/1000:,.0f} mil'.replace(',', '.'))
INICIO_ANALISE = '2025-01-01'   # padrões de dia/semana usam dados a partir daqui (2024 = implantação)

# ===== CONEXÃO =====
MODO = os.environ.get('LIAH_MODO', 'bigquery')   # 'bigquery' (bigframes) | 'offline' (lê PASTA_CACHE)
PROJETO_EXECUCAO = 'prj-sbx-data-liah-4956'    # projeto que executa e é FATURADO pelas consultas (sandbox Liah)
LOCAL_BQ = None
PASTA_DRIVE = '/content/drive/MyDrive/previsao_vendas'   # pasta no Drive com os .py (None = enviar arquivos a cada sessão)
PASTA_CACHE = 'dados/cache_dados'   # cópia local de cada consulta (permite reprocessar offline)
REUSAR_CACHE_HORAS = 12         # reaproveita consultas idênticas feitas há menos de 12 h (0 = sempre consultar o BigQuery)
PASTA_DASHBOARD = 'dashboard'   # dados_painel.js/.json vão para esta pasta (dentro de PASTA_DRIVE), junto do painel HTML
COLUNAS_MANUAIS = {}
DEDUP_ITENS = False

## 1. Conexão e dados

In [ ]:
import os, sys, importlib
if PASTA_DRIVE:
    if not os.path.isdir('/content/drive/MyDrive'):   # monta só se ainda não estiver montado (ex.: pelo ícone do Drive na barra lateral)
        from google.colab import drive
        try:
            drive.mount('/content/drive')
        except ValueError:
            drive.mount('/content/drive', force_remount=True)   # 2ª tentativa limpa uma montagem anterior travada
    os.makedirs(PASTA_DRIVE, exist_ok=True); os.chdir(PASTA_DRIVE)
    print('Usando a pasta', PASTA_DRIVE)

# Organização da pasta: só os notebooks na raiz; módulos em modulos/, histórico e cache em dados/,
# resultados em saida/, painel em dashboard/, documentação em docs/.
PASTA_MODULOS = 'modulos'
sys.dont_write_bytecode = True     # não cria a pasta __pycache__ no Drive
os.makedirs(PASTA_MODULOS, exist_ok=True)
for a in ('liah_dados.py', 'previsao_modelos.py'):   # compatibilidade: módulo esquecido na raiz vai para modulos/
    if os.path.exists(a) and not os.path.exists(os.path.join(PASTA_MODULOS, a)):
        os.replace(a, os.path.join(PASTA_MODULOS, a))
faltando = [a for a in ('liah_dados.py', 'previsao_modelos.py') if not os.path.exists(os.path.join(PASTA_MODULOS, a))]
if faltando:
    from google.colab import files
    print(f'Envie os módulos (vão para {PASTA_MODULOS}/):', faltando)
    for nome, conteudo in files.upload().items():
        with open(os.path.join(PASTA_MODULOS, nome), 'wb') as f:
            f.write(conteudo)
sys.path.insert(0, os.path.abspath(PASTA_MODULOS))
import liah_dados as ld, previsao_modelos as pm
importlib.reload(ld); importlib.reload(pm)

if MODO == 'bigquery':
    try:
        from google.colab import auth
        auth.authenticate_user()     # login Google com acesso ao projeto
    except ImportError:
        pass                         # fora do Colab: usa as credenciais padrão (gcloud auth application-default login)

fonte = ld.Fonte(MODO, PASTA_CACHE, PROJETO_EXECUCAO, LOCAL_BQ, reusar_cache_horas=REUSAR_CACHE_HORAS)
cols = ld.detectar_colunas(fonte, COLUNAS_MANUAIS)
print(f'Modo: {MODO} | consultas executadas e faturadas em: {PROJETO_EXECUCAO} | tabelas lidas de: {ld.PROJETO}')
print('Colunas detectadas nas tabelas:')
display(cols.resumo())

In [ ]:
mensal = fonte.df(ld.sql_mensal(cols, DEDUP_ITENS), 'mensal', datas=('ds', 'ultima_data')).sort_values('ds').set_index('ds')
mensal['mes_completo'] = mensal['mes_completo'].astype(str).str.lower().isin(['true', '1', '1.0'])
diaria = fonte.df(ld.sql_diaria(cols, DEDUP_ITENS), 'diaria').sort_values('ds').set_index('ds')
divisao = fonte.df(ld.sql_divisao(cols), 'divisao')
impacto = fonte.df(ld.sql_impacto_limpeza(), 'impacto_limpeza')
canal_m = fonte.df(ld.sql_canal(cols), 'canal')
FIM = diaria.index.max()
completos = mensal[mensal['mes_completo']]
print(f'Dados de {diaria.index.min():%d/%m/%Y} a {FIM:%d/%m/%Y} | meses completos até {completos.index.max():%m/%Y}')

## 2. Como estamos agora
Janelas móveis comparadas ao mesmo período do ano anterior.

In [ ]:
def janela(dias):
    atual = diaria.loc[FIM - pd.Timedelta(days=dias - 1):FIM]
    ano_ant = diaria.loc[FIM - pd.Timedelta(days=dias - 1) - pd.DateOffset(years=1):FIM - pd.DateOffset(years=1)]
    anterior = diaria.loc[FIM - pd.Timedelta(days=2 * dias - 1):FIM - pd.Timedelta(days=dias)]
    r = {}
    for nome, df in [('atual', atual), ('ano anterior', ano_ant), ('período anterior', anterior)]:
        r[nome] = {'pedidos': df['pedidos'].sum(), 'faturamento': df['faturamento'].sum(),
                   'ticket': df['faturamento'].sum() / df['pedidos'].sum()}
    t = pd.DataFrame(r).T
    t.loc['var vs ano ant. %'] = (t.loc['atual'] / t.loc['ano anterior'] - 1) * 100
    t.loc['var vs período ant. %'] = (t.loc['atual'] / t.loc['período anterior'] - 1) * 100
    return t.round(1)

for n in (30, 90):
    print(f'\n=== Últimos {n} dias ==='); display(janela(n))

mes_atual = FIM.to_period('M').to_timestamp()
a = diaria.loc[mes_atual:FIM]; b = diaria.loc[mes_atual - pd.DateOffset(years=1):FIM - pd.DateOffset(years=1)]
print(f"\nMês atual até dia {FIM.day}: {a.pedidos.sum()} pedidos ({(a.pedidos.sum()/b.pedidos.sum()-1)*100:+.0f}%), "
      f"R$ {a.faturamento.sum():,.0f} ({(a.faturamento.sum()/b.faturamento.sum()-1)*100:+.0f}%) vs mesmo período do ano anterior")

ult = completos.index.max().month
acum = completos[completos.index.month <= ult].groupby(completos[completos.index.month <= ult].index.year)[['y_total_pedidos', 'faturamento_total']].sum()
acum['ticket'] = acum['faturamento_total'] / acum['y_total_pedidos']
print(f'\nAcumulado jan–{MESES[ult-1]} por ano:'); display(acum.round(0))

## 3. Meses — ano a ano

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, col, tit in [(axes[0], 'faturamento_total', 'Faturamento por mês'), (axes[1], 'y_total_pedidos', 'Pedidos por mês')]:
    for ano, cor in CORES_ANO.items():
        s = mensal[mensal.index.year == ano]
        if s.empty: continue
        comp = s[s['mes_completo']]
        ax.plot(comp.index.month, comp[col], 'o-', color=cor, label=str(ano), lw=2)
        parc = s[~s['mes_completo']]
        if not parc.empty and not comp.empty:
            ax.plot([comp.index.month[-1], parc.index.month[0]], [comp[col].iloc[-1], parc[col].iloc[0]], 'o--', color=cor, lw=1.5)
    ax.set_xticks(range(1, 13), MESES); ax.set_title(tit + ' (tracejado = mês parcial)'); ax.legend()
axes[0].yaxis.set_major_formatter(reais)
plt.tight_layout(); plt.show()

yoy = completos[['y_total_pedidos', 'faturamento_total', 'ticket_medio']].pct_change(12).mul(100).dropna()
yoy = yoy[yoy.index >= '2026-01-01']
yoy.columns = ['Pedidos %', 'Faturamento %', 'Ticket %']
print('Variação contra o mesmo mês do ano anterior:'); display(yoy.round(1).set_axis(yoy.index.strftime('%m/%Y')))

# Sazonalidade: índice de cada mês em relação à média do ano (2025, único ano completo pós-implantação)
s25 = completos.loc['2025']
saz = (s25['y_total_pedidos'] / s25['y_total_pedidos'].mean() * 100).round(0)
saz.index = [MESES[m - 1] for m in saz.index.month]
print('\nÍndice de sazonalidade de pedidos em 2025 (100 = média do ano):'); display(saz.to_frame('índice').T)

## 4. Semanas

In [ ]:
sem = diaria.resample('W-SUN').agg({'pedidos': 'sum', 'faturamento': 'sum'})
sem = sem[(sem.index <= FIM) & (sem.index >= INICIO_ANALISE)]
sem.index = sem.index - pd.Timedelta(days=6)   # rótulo = segunda-feira
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.bar(sem.index, sem['faturamento'], width=5, color='#86b6ef', label='Faturamento da semana')
ax.plot(sem.index, sem['faturamento'].rolling(4).mean(), color='black', lw=2, label='Média móvel 4 semanas')
ax.yaxis.set_major_formatter(reais); ax.set_title('Faturamento semanal (segunda a domingo)'); ax.legend()
plt.tight_layout(); plt.show()
print('Melhores semanas:'); display(sem.sort_values('faturamento', ascending=False).head(6).round(0).set_axis(sem.sort_values('faturamento', ascending=False).head(6).index.strftime('%d/%m/%Y')))
print('Últimas 8 semanas:'); display(sem.tail(8).round(0).set_axis(sem.tail(8).index.strftime('%d/%m/%Y')))

## 5. Dias da semana e dias do mês
Índice 100 = média diária de pedidos desde `INICIO_ANALISE`.

In [ ]:
dd = diaria.loc[INICIO_ANALISE:FIM].copy()
dd['dow'] = dd.index.dayofweek; dd['dia'] = dd.index.day
media = dd['pedidos'].mean()
dow = dd.groupby('dow').agg(pedidos_dia=('pedidos', 'mean'), fat_total=('faturamento', 'sum'), ped_total=('pedidos', 'sum'))
dow['ticket'] = dow['fat_total'] / dow['ped_total']
dow['indice'] = dow['pedidos_dia'] / media * 100
dow['% faturamento'] = dow['fat_total'] / dow['fat_total'].sum() * 100
dom = dd.groupby('dia')['pedidos'].mean() / media * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].bar(DIAS, dow['indice'], color=[AZUL if v >= 100 else '#b7d3f6' for v in dow['indice']])
axes[0].axhline(100, color='gray', lw=1); axes[0].set_title('Pedidos por dia da semana (índice)')
axes[1].bar(dom.index, dom.values, color=[AZUL if v >= 100 else '#b7d3f6' for v in dom.values])
axes[1].axhline(100, color='gray', lw=1); axes[1].set_title('Pedidos por dia do mês (índice)')
plt.tight_layout(); plt.show()
display(dow[['pedidos_dia', 'indice', 'ticket', '% faturamento']].set_axis(DIAS).round(1))

faixas = pd.cut(dd['dia'], [0, 5, 10, 15, 20, 25, 31], labels=['1–5', '6–10', '11–15', '16–20', '21–25', '26–31'])
print('Índice por faixa do mês:'); display((dd.groupby(faixas)['pedidos'].mean() / media * 100).round(1).to_frame('índice').T)

# Mapa de calor dia da semana × mês
hh = dd[dd.index <= completos.index.max() + pd.offsets.MonthEnd(0)].assign(mes=lambda x: x.index.month)
hm = hh.pivot_table(index='dow', columns='mes', values='pedidos', aggfunc='mean')
fig, ax = plt.subplots(figsize=(13, 3.8))
im = ax.imshow(hm.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(hm.shape[1]), [MESES[m - 1] for m in hm.columns]); ax.set_yticks(range(7), DIAS); ax.grid(False)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        v = hm.values[i, j]
        ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=9, color='white' if v > np.nanpercentile(hm.values, 60) else 'black')
ax.set_title('Pedidos médios por dia: dia da semana × mês'); plt.colorbar(im, ax=ax, shrink=.8)
plt.tight_layout(); plt.show()

## 6. Datas especiais

In [ ]:
feriados = holidays.Brazil(years=range(2024, FIM.year + 1))
dd['feriado'] = [d in feriados for d in dd.index.date]
print(f"Feriados: {dd[dd.feriado].pedidos.mean():.1f} pedidos/dia | dias úteis comuns: {dd[(~dd.feriado) & (dd.dow < 5)].pedidos.mean():.1f}")

def black_friday(ano):  # sexta-feira seguinte à 4ª quinta-feira de novembro
    nov = pd.date_range(f'{ano}-11-01', f'{ano}-11-30'); return nov[nov.dayofweek == 3][3] + pd.Timedelta(days=1)

periodos = []
for ano in range(2024, FIM.year + 1):
    bf = black_friday(ano)
    periodos.append((f'Black Friday {ano} (±3 dias)', bf - pd.Timedelta(days=3), bf + pd.Timedelta(days=3), pd.Timestamp(f'{ano}-11-01'), bf - pd.Timedelta(days=4)))
    periodos.append((f'Natal e Ano-Novo {ano}', pd.Timestamp(f'{ano}-12-20'), pd.Timestamp(f'{ano + 1}-01-02'), pd.Timestamp(f'{ano}-12-01'), pd.Timestamp(f'{ano}-12-19')))
linhas = []
for nome, i, f, ri, rf in periodos:
    if f > FIM: continue
    a, b = diaria.loc[i:f, 'pedidos'].mean(), diaria.loc[ri:rf, 'pedidos'].mean()
    linhas.append({'período': nome, 'pedidos/dia': round(a, 1), 'referência': round(b, 1), 'variação %': round((a / b - 1) * 100)})
display(pd.DataFrame(linhas))

## 7. Rankings

In [ ]:
top_m = completos.loc[INICIO_ANALISE:].sort_values('faturamento_total', ascending=False).head(6)[['faturamento_total', 'y_total_pedidos', 'ticket_medio']]
top_d = diaria.loc[INICIO_ANALISE:].sort_values('faturamento', ascending=False).head(10)[['pedidos', 'faturamento']]
top_d.insert(0, 'dia da semana', [DIAS[d] for d in top_d.index.dayofweek])
print('Melhores meses:'); display(top_m.round(0).set_axis(top_m.index.strftime('%m/%Y')))
print('Melhores dias:'); display(top_d.round(0).set_axis(top_d.index.strftime('%d/%m/%Y')))

## 8. Ticket, mix e pedidos não concluídos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].plot(mensal.index, mensal['ticket_medio'], 'o-', color=AZUL, lw=2); axes[0].set_title('Ticket médio por mês (R$)')
axes[1].plot(mensal.index, mensal['media_itens_por_pedido'], 'o-', color=LARANJA, lw=2); axes[1].set_title('Itens por pedido')
plt.tight_layout(); plt.show()

o = divisao[divisao.ds >= '2026-05-01'].pivot_table(index='ds', columns='origem', values='faturamento', aggfunc='sum').fillna(0)
o.index = o.index.strftime('%m/%Y')
ax = o.plot(kind='bar', stacked=True, color=[{'1P': AZUL, '3P': LARANJA}.get(c, CINZA) for c in o.columns], figsize=(9, 4), rot=0)
ax.yaxis.set_major_formatter(reais); ax.set_title('Faturamento por origem'); plt.tight_layout(); plt.show()
print('% do faturamento por origem:'); display((o.div(o.sum(axis=1), axis=0) * 100).round(1))

im_ = impacto.pivot_table(index='ds', columns='situacao', values='pedidos', aggfunc='sum').fillna(0)
nc = im_.drop(columns='VALIDO').sum(axis=1) / im_.sum(axis=1) * 100
ax = nc.plot(marker='o', color=LARANJA, figsize=(13, 3.8)); ax.set_ylim(0, None)
ax.set_title('Pedidos não concluídos (% dos criados): cancelados, PIX não pago, aguardando, testes'); plt.tight_layout(); plt.show()

## 9. Granularidades: canal, origem, região/UF, grupo, subgrupo e marca
Cada corte é um `groupby` local dos cubos semanais (já agregados no BigQuery, baixados uma vez). Pedidos extremos ficam de fora para não
distorcer marca/produto.

**Ruído por nível:** o *coeficiente de variação* (CV) semanal mede quanto a série oscila em torno da própria média.
Quanto mais fino o corte, maior o CV — e mais difícil prever. É por isso que o notebook de previsão escolhe um modelo por nível.

In [ ]:
# Dois cubos semanais (produto e geográfico), baixados UMA vez e já agregados no BigQuery. Cada granularidade
# vira um groupby local, sem nova leitura das tabelas. Separados para caber na memória do Colab.
cubos = ld.Cubos(fonte, cols, DEDUP_ITENS)
ult_sem = ld.ultima_semana_completa(FIM)
CORTES = {'Canal': 'canal', 'Origem': 'origem', 'Região': 'regiao', 'UF': 'uf', 'Grupo': 'grupo', 'Subgrupo': 'subgrupo',
          'Marca': 'marca', 'Produto': 'produto_nome'}
MES = {}
ruido = {}
semanal_tot = None
for nome_c, col in CORTES.items():
    g = cubos.agregar([col], ['faturamento', 'unidades'])
    g['semana'] = ld._sem_fuso(g['semana'])
    g = g[(g['semana'] >= INICIO_ANALISE) & (g['semana'] <= ult_sem)]
    if g[col].astype(str).str.startswith('SEM_').all():
        print(f'{nome_c}: sem informação na base — pulado'); continue
    piv = g.pivot_table(index='semana', columns=col, values='faturamento', aggfunc='sum').fillna(0)
    semanal_tot = piv.sum(axis=1)
    peso = piv.sum() / piv.values.sum()
    cv = piv.std() / piv.mean().replace(0, np.nan)
    ruido[nome_c] = {'séries': piv.shape[1], 'CV semanal médio (ponderado)': np.nansum(cv * peso)}
    MES[nome_c] = g.assign(mes=g['semana'].dt.to_period('M').dt.to_timestamp()).groupby(['mes', col])['faturamento'].sum().unstack(fill_value=0)
ruido = pd.DataFrame(ruido).T
if semanal_tot is not None:
    ruido.loc['Total'] = [1, semanal_tot.std() / semanal_tot.mean()]
ruido = ruido.sort_values('CV semanal médio (ponderado)')
ax = ruido['CV semanal médio (ponderado)'].plot.barh(color=AZUL, figsize=(9, 4))
ax.set_title('Oscilação semanal por nível (CV): quanto mais fino, mais ruído'); plt.tight_layout(); plt.show()
display(ruido.round(2))

In [ ]:
mostrar = [c for c in ('Canal', 'Região', 'Grupo', 'Marca') if c in MES]
fig, axes = plt.subplots(len(mostrar), 2, figsize=(16, 3.6 * len(mostrar)), squeeze=False)
for (ax1, ax2), c in zip(axes, mostrar):
    m = MES[c][MES[c].index < FIM.to_period('M').to_timestamp()]
    top = m.sum().sort_values(ascending=False).index[:8]
    m = m[top].assign(Outros=m.drop(columns=top).sum(axis=1)) if m.shape[1] > 8 else m[top]
    m.plot(ax=ax1, marker='o', ms=3); ax1.yaxis.set_major_formatter(reais); ax1.set_title(f'{c}: faturamento mensal (sem extremos)')
    ax1.legend(fontsize=7, ncol=2)
    (m.div(m.sum(axis=1), axis=0) * 100).plot.area(ax=ax2, alpha=.85, lw=0, cmap='tab20'); ax2.set_ylim(0, 100)
    ax2.set_title(f'{c}: participação %'); ax2.legend(fontsize=7, ncol=2, loc='upper left')
plt.tight_layout(); plt.show()

# crescimento: últimos 3 meses completos × mesmos meses do ano anterior, por corte
ult3 = completos.index[-3:]
for c in mostrar:
    m = MES[c]
    a, b = m.reindex(ult3).sum(), m.reindex(ult3 - pd.DateOffset(years=1)).sum()
    t = pd.DataFrame({'últimos 3 meses': a, 'mesmos meses ano ant.': b, 'var %': (a / b.replace(0, np.nan) - 1) * 100,
                      'participação %': a / a.sum() * 100}).sort_values('últimos 3 meses', ascending=False).head(10)
    print(f'\n{c}:'); display(t.round(1))

## 10. Gerar os dados do painel
Gera o arquivo que alimenta o painel HTML (`painel_vendas_liah.html`):
- `dados_painel.js` → gravado direto na pasta `dashboard` (`PASTA_DASHBOARD`) — deixe o `painel_vendas_liah.html` nessa pasta e abra o HTML no navegador (funciona direto do computador).
- `dados_painel.json` → mesmo conteúdo em JSON; pode ser carregado pelo botão "Carregar dados" do painel ou usado por outro sistema.

In [ ]:
import json
from dateutil.easter import easter

def _r(v, n=2):
    return None if pd.isna(v) else round(float(v), n)

painel = {'gerado_em': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'), 'fim': FIM.strftime('%Y-%m-%d')}

# Mensal
painel['mensal'] = [dict(m=ds.strftime('%Y-%m'), ped=int(r.y_total_pedidos), fat=_r(r.faturamento_total),
                         tk=_r(r.ticket_medio), itens=_r(r.media_itens_por_pedido), pu=_r(r.preco_medio_unidade),
                         completo=bool(r.mes_completo)) for ds, r in mensal.iterrows()]

# Semanal (segunda a domingo), semanas completas
w = diaria.resample('W-SUN').agg({'pedidos': 'sum', 'faturamento': 'sum'})
w = w[w.index <= FIM]
painel['semanal'] = [dict(s=(i - pd.Timedelta(days=6)).strftime('%Y-%m-%d'), ped=int(r.pedidos), fat=_r(r.faturamento))
                     for i, r in w.iterrows()]

# Dia da semana, dia do mês, faixas do mês
painel['inicio_padroes'] = pd.Timestamp(INICIO_ANALISE).strftime('%Y-%m-%d')
painel['dow'] = [dict(d=int(i), ped=_r(r.pedidos_dia, 1), tk=_r(r.ticket, 1), idx=_r(r.indice, 1), share=_r(r['% faturamento'], 1))
                 for i, r in dow.iterrows()]
painel['dia_mes'] = [dict(dia=int(i), ped=_r(dd[dd.dia == i].pedidos.mean(), 1), idx=_r(v, 1)) for i, v in dom.items()]
fx = dd.groupby(faixas)['pedidos'].mean() / media * 100
painel['faixas_mes'] = [dict(faixa=str(k), idx=_r(v, 1)) for k, v in fx.items()]

# Mapa de calor (linhas = seg..dom, colunas = jan..dez; null se não houver dado)
hm12 = hm.reindex(index=range(7), columns=range(1, 13))
painel['heat'] = [[_r(v, 1) for v in row] for row in hm12.values.tolist()]

# Feriados e datas especiais
painel['feriado'] = dict(media_feriado=_r(dd[dd.feriado].pedidos.mean(), 1),
                         media_util=_r(dd[(~dd.feriado) & (dd.dow < 5)].pedidos.mean(), 1),
                         media_geral=_r(dd.pedidos.mean(), 1))
def _media(i, f):
    x = diaria.loc[i:f, 'pedidos']; return x.mean() if len(x) else np.nan
esp = []
for ano in range(diaria.index.min().year, FIM.year + 1):
    bf = black_friday(ano); pascoa = pd.Timestamp(easter(ano)); carnaval = pascoa - pd.Timedelta(days=47)
    janelas = [
        (f'Carnaval {ano}', carnaval - pd.Timedelta(days=3), carnaval + pd.Timedelta(days=1), carnaval - pd.Timedelta(days=17), carnaval - pd.Timedelta(days=4)),
        (f'Semana Santa {ano}', pascoa - pd.Timedelta(days=2), pascoa + pd.Timedelta(days=1), pascoa - pd.Timedelta(days=16), pascoa - pd.Timedelta(days=3)),
        (f'Black Friday {ano} (±3 dias)', bf - pd.Timedelta(days=3), bf + pd.Timedelta(days=3), pd.Timestamp(f'{ano}-11-01'), bf - pd.Timedelta(days=4)),
        (f'Natal e Ano-Novo {ano}', pd.Timestamp(f'{ano}-12-20'), pd.Timestamp(f'{ano + 1}-01-02'), pd.Timestamp(f'{ano}-12-01'), pd.Timestamp(f'{ano}-12-19')),
    ]
    for nome, i, f, ri, rf in janelas:
        if f > FIM or ri < diaria.index.min(): continue
        a, b = _media(i, f), _media(ri, rf)
        if np.isnan(a) or np.isnan(b) or b < 10: continue   # ignora períodos com volume muito baixo (implantação)
        esp.append(dict(nome=nome, inicio=i.strftime('%Y-%m-%d'), media=_r(a, 1), ref=_r(b, 1), var=_r((a / b - 1) * 100, 0)))
painel['especiais'] = sorted(esp, key=lambda e: e['inicio'])

# Rankings
td = diaria.loc[INICIO_ANALISE:].sort_values('faturamento', ascending=False).head(8)
painel['top_dias'] = [dict(d=i.strftime('%Y-%m-%d'), ped=int(r.pedidos), fat=_r(r.faturamento, 0)) for i, r in td.iterrows()]
ts = sem.sort_values('faturamento', ascending=False).head(6)
painel['top_semanas'] = [dict(s=i.strftime('%Y-%m-%d'), ped=int(r.pedidos), fat=_r(r.faturamento, 0)) for i, r in ts.iterrows()]

# Janelas recentes e acumulado do ano
def _jan(n):
    t = janela(n)
    return dict(ped=int(t.loc['atual', 'pedidos']), fat=_r(t.loc['atual', 'faturamento'], 0), tk=_r(t.loc['atual', 'ticket'], 1),
                ped_ly=int(t.loc['ano anterior', 'pedidos']), fat_ly=_r(t.loc['ano anterior', 'faturamento'], 0),
                tk_ly=_r(t.loc['ano anterior', 'ticket'], 1),
                ped_prev=int(t.loc['período anterior', 'pedidos']), fat_prev=_r(t.loc['período anterior', 'faturamento'], 0))
painel['j30'], painel['j90'] = _jan(30), _jan(90)
a = diaria.loc[mes_atual:FIM]; b = diaria.loc[mes_atual - pd.DateOffset(years=1):FIM - pd.DateOffset(years=1)]
painel['mtd'] = dict(mes=mes_atual.strftime('%Y-%m'), ped=int(a.pedidos.sum()), fat=_r(a.faturamento.sum(), 0),
                     ped_ly=int(b.pedidos.sum()), fat_ly=_r(b.faturamento.sum(), 0))
ano_at = completos.index.max().year
ytd = completos[(completos.index.month <= ult) & completos.index.year.isin([ano_at, ano_at - 1])].groupby(lambda d: d.year)[['y_total_pedidos', 'faturamento_total']].sum()
painel['ytd'] = dict(ano=int(ano_at), mes_fim=int(ult),
                     atual=dict(ped=int(ytd.loc[ano_at, 'y_total_pedidos']), fat=_r(ytd.loc[ano_at, 'faturamento_total'], 0)),
                     anterior=dict(ped=int(ytd.loc[ano_at - 1, 'y_total_pedidos']), fat=_r(ytd.loc[ano_at - 1, 'faturamento_total'], 0)) if ano_at - 1 in ytd.index else None)

# Opcionais: pedidos não concluídos e origem 1P/3P
painel['nao_concluidos'] = [dict(m=i.strftime('%Y-%m'), pct=_r(v, 1)) for i, v in nc.items()] if 'nc' in globals() else []
painel['origem'] = ([dict(m=pd.to_datetime(i, format='%m/%Y').strftime('%Y-%m'), **{k: _r(v, 0) for k, v in r.items()}) for i, r in o.iterrows()] if 'o' in globals() else [])
# Novo: mix mensal por corte (canal, grupo, marca...) — chaves extras, não afetam o painel atual
painel['mix'] = {c: [dict(m=i.strftime('%Y-%m'), **{str(k): _r(v, 0) for k, v in r.items()}) for i, r in MES[c].iterrows()] for c in MES}
painel['ruido_por_nivel'] = {k: _r(v, 3) for k, v in ruido['CV semanal médio (ponderado)'].items()}

os.makedirs(PASTA_DASHBOARD, exist_ok=True)
arq_json = os.path.join(PASTA_DASHBOARD, 'dados_painel.json')
arq_js = os.path.join(PASTA_DASHBOARD, 'dados_painel.js')
with open(arq_json, 'w', encoding='utf-8') as f:
    json.dump(painel, f, ensure_ascii=False)
with open(arq_js, 'w', encoding='utf-8') as f:
    f.write('window.DADOS_PAINEL = ' + json.dumps(painel, ensure_ascii=False) + ';\n')
print(f"Gerados {arq_js} e {arq_json} (dados até {FIM:%d/%m/%Y})")
if not PASTA_DRIVE:   # sem Drive: baixa no computador
    try:
        from google.colab import files
        files.download(arq_js); files.download(arq_json)
    except ImportError:
        pass

In [ ]:
fonte.resumo_custo()   # consultas feitas no BigQuery × reaproveitadas do cache